# What you will learn

This 90-minute workshop uses Python to carry out a reproducible analysis of raw questionnaire data. It assumes familiarity with statistical reasoning, not with programming.

By the end, you will be able to:

- download a public dataset and inspect it with pandas;
- clean response codes and make an explicit missing-data decision;
- compute Big Five scale scores from a documented scoring key;
- test a statistical question and produce a publication-ready plot;
- recognise data leakage in models and analyses;
- ask an LLM for code without uploading respondent-level rows;
- distinguish pattern-based masking, automated PII detection, word counts, and semantic embeddings.

**Version:** participant notebook. Complete the short exercises marked **Your task**.

## Schedule

| Time | Module |
|---:|---|
| 0-5 | Setup and reproducible download |
| 5-15 | pandas inspection and selection |
| 15-28 | Cleaning and missing-data decisions |
| 28-43 | Big Five scoring |
| 43-55 | Statistical question and visualisation |
| 55-62 | Data leakage |
| 62-74 | LLM workflow without sharing rows |
| 74-82 | PII masking and automated detection |
| 82-90 | Bag of Words and sentence embeddings |

The public workshop data are not a model for handling confidential research records. For real data, follow the consent form, institutional policy, data-processing agreements, and the approved computing environment.


## 1. Setup and data download

Run the cell below to prepare your own Colab session. It downloads all files from the workshop repository, so helper modules such as `nbutils.py` or `nbutils/` at the repository root can be imported with `import nbutils`. Rerunning the cell keeps the existing repository files in this runtime.

The source is the [Big Five Personality Test dataset on Kaggle](https://www.kaggle.com/datasets/tunguz/big-five-personality-test), originally collected by the Open-Source Psychometrics Project. The cell also downloads `data-final.csv` to the temporary disk attached to this Colab runtime. The dataset is not read from your personal computer or from the workshop repository.

The complete file is large. For a live workshop we read 5,000 rows and only the columns needed below. This is a convenience sample for learning a workflow, not a representative population sample.


In [ ]:
%pip -q install kagglehub

import io
import os
import sys
import zipfile
from pathlib import Path
from urllib.request import urlopen

# Download all repository files once per Colab runtime, including helper modules.
if "google.colab" in sys.modules:
    REPO_DIR = Path("/content/Data-Analysis-Workshop-main")
    if not REPO_DIR.is_dir():
        archive_url = "https://github.com/bajon1/Data-Analysis-Workshop/archive/refs/heads/main.zip"
        with urlopen(archive_url, timeout=60) as response:
            with zipfile.ZipFile(io.BytesIO(response.read())) as archive:
                archive.extractall(REPO_DIR.parent)
else:
    REPO_DIR = Path.cwd()
    if REPO_DIR.name == "notebooks":
        REPO_DIR = REPO_DIR.parent

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"Workshop repository: {REPO_DIR}")

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

DATASET_HANDLE = "tunguz/big-five-personality-test"
downloaded = Path(kagglehub.dataset_download(
    DATASET_HANDLE,
    path="IPIP-FFM-data-8Nov2018/data-final.csv",
))

if downloaded.is_file():
    DATA_FILE = downloaded
else:
    matches = list(downloaded.rglob("data-final.csv"))
    if not matches:
        raise FileNotFoundError("Kaggle download completed, but data-final.csv was not found")
    DATA_FILE = matches[0]

PREFIXES = ["EXT", "AGR", "CSN", "EST", "OPN"]
ITEM_COLUMNS = [f"{prefix}{number}" for prefix in PREFIXES for number in range(1, 11)]
USE_COLUMNS = ITEM_COLUMNS + ["country", "IPC", "testelapse"]

df_raw = pd.read_csv(DATA_FILE, sep="	", usecols=USE_COLUMNS, nrows=5_000)
df_raw.shape


## 2. Pandas: inspect and select

A pandas `DataFrame` is a table with named columns. The first diagnostic pass answers four questions: What are the rows and columns? Which columns are numeric? Are values missing? Are the observed ranges compatible with the codebook?


In [ ]:
df_raw.head()


In [ ]:
df_raw.info()


In [ ]:
df_raw[ITEM_COLUMNS + ["testelapse"]].describe().T.loc[
    ["EXT1", "AGR1", "CSN1", "EST1", "OPN1", "testelapse"]
].round(2)


In [ ]:
df_raw[["country", "IPC"]].head()

df_raw["country"].value_counts(dropna=False).head(10)

df_raw.loc[df_raw["testelapse"] < 60, ["country", "testelapse"]].head()


### Your task

Select records with country code `PL` and display only `country` and `testelapse`. The two pairs of brackets have different roles: the first filters rows, the second selects columns.


In [ ]:
polish_rows = df_raw.loc[
    df_raw["country"] == "___",
    ["___", "___"],
]
polish_rows.head()


Pandas documentation is part of the workflow, not an afterthought: [`DataFrame.loc`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html), [`DataFrame.info`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html), and [`DataFrame.describe`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html).


## 3. Cleaning and missing data

The questionnaire response scale is 1-5, but the file uses `0` for an unanswered item. A zero is therefore not a low score: it is a missing response. This distinction comes from the codebook, not from pandas.

Python is useful here because the same documented operations can be applied to all 50 items and rerun when new data arrive. The original `df_raw` remains unchanged.


In [ ]:
quality_summary = pd.Series({
    "rows": len(df_raw),
    "exact duplicate rows": df_raw.duplicated().sum(),
    "zero response codes": df_raw[ITEM_COLUMNS].eq(0).sum().sum(),
    "non-positive completion times": df_raw["testelapse"].le(0).sum(),
})
quality_summary.to_frame("count")


### Your task

Replace the missing-response code in all questionnaire items at once. Use `0`, not the string `"0"`.


In [ ]:
df_clean = df_raw.drop_duplicates().copy()
df_clean[ITEM_COLUMNS] = df_clean[ITEM_COLUMNS].replace("___", np.nan)
df_clean["testelapse"] = df_clean["testelapse"].where(df_clean["testelapse"] > 0)

df_clean[ITEM_COLUMNS].isna().sum().sort_values(ascending=False).head()


### Imputation is an analytic decision

Median imputation is easy to automate, but it changes the response distribution and can attenuate uncertainty. The following copy demonstrates mass imputation; it is not used for the primary scale scores. In the scoring section we instead require at least 8 observed items out of 10 and average the available responses.


In [ ]:
df_imputed = df_clean.copy()
item_medians = df_imputed[ITEM_COLUMNS].median()
df_imputed[ITEM_COLUMNS] = df_imputed[ITEM_COLUMNS].fillna(item_medians)

pd.DataFrame({
    "missing before": df_clean[ITEM_COLUMNS].isna().sum(),
    "missing after": df_imputed[ITEM_COLUMNS].isna().sum(),
}).head()


Deletion, single imputation, multiple imputation, and partial-scale scoring answer different questions. The choice should follow the instrument manual and analysis plan; a convenient method is not automatically a defensible one.


## 4. Scoring the Big Five

The five prefixes represent Extraversion (`EXT`), Agreeableness (`AGR`), Conscientiousness (`CSN`), Emotional Stability (`EST`), and Openness/Intellect (`OPN`). Each scale contains positively and negatively keyed items.

For a 1-5 item, reverse scoring is `6 - response`. The item directions below follow the [official IPIP Big-Five key](https://ipip.ori.org/newBigFive5broadKey.htm) and the [IPIP scoring instructions](https://ipip.ori.org/newScoringInstructions.htm).


In [ ]:
SCALES = {
    "EXT": [f"EXT{i}" for i in range(1, 11)],
    "AGR": [f"AGR{i}" for i in range(1, 11)],
    "CSN": [f"CSN{i}" for i in range(1, 11)],
    "EST": [f"EST{i}" for i in range(1, 11)],
    "OPN": [f"OPN{i}" for i in range(1, 11)],
}

REVERSE_ITEMS = {
    "EXT2", "EXT4", "EXT6", "EXT8", "EXT10",
    "AGR1", "AGR3", "AGR5", "AGR7",
    "CSN2", "CSN4", "CSN6", "CSN8",
    "EST1", "EST3", "EST5", "EST6", "EST7", "EST8", "EST9", "EST10",
    "OPN2", "OPN4", "OPN6",
}


### One respondent: a loop and a condition

This deliberately uses ordinary Python before the vectorised pandas version. It makes the scoring rule visible and transferable to another questionnaire.


In [ ]:
person = df_clean.loc[0]
trait = "EXT"
scored_answers = []

for item in SCALES[trait]:
    answer = person[item]
    if item in []:  # replace [] with the documented set of reverse-keyed items
        answer = 6 - answer
    scored_answers.append(answer)

np.nanmean(scored_answers)


### All respondents

Work on a copy so the original responses stay available for auditing. A score is retained only when at least 8 of 10 items were answered.


In [ ]:
df_scored = df_clean.copy()
df_scored[list(REVERSE_ITEMS)] = 6 - df_scored[list(REVERSE_ITEMS)]

minimum_answered = 99  # replace with the workshop rule
for scale, items in SCALES.items():
    answered = df_clean[items].notna().sum(axis=1)
    df_scored[f"{scale}_score"] = df_scored[items].mean(axis=1).where(
        answered >= minimum_answered
    )

SCORE_COLUMNS = [f"{scale}_score" for scale in SCALES]
df_scored[SCORE_COLUMNS].describe().round(2)


In [ ]:
missing_scores = df_scored[SCORE_COLUMNS].isna().sum().to_frame("missing scale scores")
display(missing_scores)

df_scored.to_csv("big_five_scored_workshop.csv", index=False)


## 5. Statistical question and visualisation

**Question:** Are Extraversion and Agreeableness monotonically associated in this sample?

We use Spearman's rank correlation because the exercise asks about a monotonic association and does not require a bivariate-normal model. The coefficient is the effect-size estimate. With thousands of observations, a small effect can have a very small p-value; interpret magnitude and uncertainty, not only a threshold.

You already know how to choose a plot. Here the focus is the matplotlib pattern: create axes, draw, label, and save.


In [ ]:
analysis_data = df_scored[["EXT_score", "___"]].dropna()

result = stats.spearmanr(
    analysis_data["EXT_score"],
    analysis_data["___"],
)
summary = pd.Series({"Spearman rho": result.statistic, "p-value": result.pvalue, "n": len(analysis_data)})
display(summary)

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(analysis_data["EXT_score"], analysis_data["___"], alpha=0.15, s=16)
ax.set(xlabel="Extraversion score", ylabel="Agreeableness score",
       title="Extraversion and Agreeableness")
fig.tight_layout()
fig.savefig("extraversion_agreeableness.png", dpi=200, bbox_inches="tight")
plt.show()


A defensible report would add a confidence interval, discuss measurement reliability and missingness, and state that this internet sample is self-selected. Statistical verification does not repair a weak design.


## 6. Data leakage

Data leakage means that information unavailable at the intended decision point, or information from the evaluation data, enters the analysis. It produces results that are easier than the real problem.

Common cases:

- imputing, scaling, or selecting variables on the full dataset before a train/test split;
- putting the outcome, or a variable computed from it, among the predictors;
- putting repeated measurements from one participant on both sides of a split;
- testing a scale against one of the items used to construct that scale and presenting the association as independent evidence;
- choosing hypotheses after inspecting many results without accounting for that search.

The next comparison is not a prediction model. It exposes a circular analysis: `EXT1` is part of `EXT_score`, so their association is partly built in.


In [ ]:
leakage_demo = pd.Series({
    "circular: EXT score vs its own EXT1 item": stats.spearmanr(
        df_scored["EXT_score"], df_scored["EXT1"], nan_policy="omit"
    ).statistic,
    "non-overlapping scales: EXT vs AGR": stats.spearmanr(
        df_scored["EXT_score"], df_scored["AGR_score"], nan_policy="omit"
    ).statistic,
})
leakage_demo.to_frame("Spearman rho")


For predictive work, split first and put learned preprocessing inside a pipeline:

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)
model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), estimator)
model.fit(X_train, y_train)  # preprocessing is learned from training rows only
model.score(X_test, y_test)  # the test set is used for final evaluation
```

The exact split must respect the design: grouped by participant for repeated measures, and often by site or time for deployment questions.


## 7. Using an LLM without sharing rows

Generating pandas or matplotlib code rarely requires respondent-level data. A safer prompt contains the table name, column names, data types, missingness summaries, and the analytic task. It does not contain `head()`, copied rows, names, free text, dates of birth, IDs, or rare combinations.

The function below builds a structure-only description. Even summaries can be sensitive in a small study, so apply institutional disclosure rules before sending them to an external service.


In [ ]:
def dataframe_schema(data: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "column": data.columns,
        "dtype": data.dtypes.astype(str).values,
        "missing_percent": (data.isna().mean().mul(100).round(1)).values,
    })

schema = dataframe_schema(df_clean[ITEM_COLUMNS + ["country", "testelapse"]])
schema.head()


### Your task

Build a prompt that asks for code to compute the mean and standard deviation of each `EXT` item. The LLM should return code only and must not require access to the data.


In [ ]:
task = "___"

safe_prompt = f"""I work in Google Colab with a pandas DataFrame named df_clean.
Here is its schema and missingness summary; no respondent rows are included:
{schema.to_csv(index=False)}

Task: {task}
Use pandas. Do not request or invent data. Return code only."""

safe_prompt


### Working patterns beyond repeatedly pasting tables into chat

- Keep computation in the notebook and use the LLM only to draft code from a schema.
- Use an institution-approved assistant or API endpoint with the required contract, retention settings, and access controls.
- Use a local model for tasks that policy permits but an external service does not.
- Connect a notebook to an approved API programmatically so the transformation is reproducible; keep keys in Colab Secrets or an environment variable, never in the notebook.
- Validate generated code against library documentation and a non-sensitive test table before running it on research data.

No integration choice overrides consent, ethics approval, information-security policy, or a data-processing agreement.


In [ ]:
pd.DataFrame.fillna?


## 8. Regex masking and Automated PII detection

Regular expressions are useful for stable patterns such as e-mail addresses and phone numbers. They do not understand that a phrase is a person's name or a location.


In [ ]:
import re

example_notes = pd.Series([
    "Contact: anna.nowak@example.org, phone +48 500 600 700.",
    "John Smith said that he lives in Krakow.",
], name="original")

masked = example_notes.str.replace(
    r"[\w.+-]+@[\w-]+\.[\w.-]+", "[EMAIL]", regex=True
).str.replace(
    r"\+?\d[\d -]{7,}\d", "[PHONE]", regex=True
)

pd.DataFrame({"original": example_notes, "regex_masked": masked})


### Optional demonstration: Microsoft Presidio

[Presidio](https://microsoft.github.io/presidio/) combines recognisers based on patterns, checksums, context, and named-entity models. This is a library-based demonstration, not a home-made anonymiser. Run the next two cells only if time and network access allow.


In [ ]:
%pip -q install presidio-analyzer presidio-anonymizer
!python -m spacy download en_core_web_sm


In [ ]:
from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_anonymizer import AnonymizerEngine

configuration = {
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "en", "model_name": "en_core_web_sm"}],
}
nlp_engine = NlpEngineProvider(nlp_configuration=configuration).create_engine()
analyzer = AnalyzerEngine(nlp_engine=nlp_engine, supported_languages=["en"])
anonymizer = AnonymizerEngine()

text = "John Smith said that he lives in Krakow. Write to john@example.org."
detections = analyzer.analyze(
    text=text,
    language="en",
    entities=["PERSON", "LOCATION", "EMAIL_ADDRESS"],
)
anonymizer.anonymize(text=text, analyzer_results=detections).text


Automated detection can miss nicknames, street addresses, workplaces, indirect identifiers, and language-specific forms. Presidio explicitly does not guarantee that all sensitive information will be found. Human review and data minimisation remain necessary; when disclosure risk cannot be controlled, do not send the text.


## 9. Bag of Words and sentence embeddings

These are two compact ways to turn text into numbers.

**Bag of Words** counts tokens. It is transparent and useful for word statistics, but ignores most meaning and word order. **Sentence embeddings** use a pretrained model to place whole texts in a vector space where semantically related statements can be close even when they use different words.

The examples use invented sentences, not research notes.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

invented_notes = [
    "The participant reports difficulty sleeping and morning fatigue.",
    "Sleep improved, but tiredness is still present after waking.",
    "The participant enjoys social meetings and group activities.",
]

vectorizer = CountVectorizer(stop_words="english")
bag = vectorizer.fit_transform(invented_notes)
word_counts = pd.DataFrame(
    bag.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=["note_1", "note_2", "note_3"],
)
word_counts


In [ ]:
# Change ngram_range so the vocabulary can also contain two-word phrases.
vectorizer_with_phrases = CountVectorizer(stop_words="english", ngram_range=(1, 1))
phrases = vectorizer_with_phrases.fit_transform(invented_notes)
vectorizer_with_phrases.get_feature_names_out()


The final optional cell downloads a small pretrained sentence-transformer. Compare the similarity matrix with the exact word counts above: the two sleep-related notes should be closer to each other than to the social-activity note.


In [ ]:
%pip -q install sentence-transformers


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = embedding_model.encode(invented_notes, normalize_embeddings=True)
semantic_similarity = embeddings @ embeddings.T

pd.DataFrame(
    semantic_similarity,
    index=["note_1", "note_2", "note_3"],
    columns=["note_1", "note_2", "note_3"],
).round(2)


## Closing summary

The reusable workflow is:

1. preserve the raw file and inspect its structure;
2. translate the codebook into explicit cleaning rules;
3. keep scale scoring in a documented, auditable configuration;
4. pair statistical tests with effect sizes, assumptions, and design limitations;
5. generate plots locally;
6. separate LLM code assistance from respondent-level data;
7. treat automated de-identification as risk reduction, not a guarantee;
8. choose text representations according to the question: counts for transparent frequency statistics, embeddings for semantic proximity.

To adapt the notebook to another questionnaire, replace the item columns, scale mapping, reverse-key set, completeness rule, and instrument-specific documentation. Do not copy a scoring rule without checking the manual and licence.
